# 06 — embedded chunk `6` (aboutTheJob)

Fixture: `tests/fixtures/linkedin/about_the_job/embedded_chunk_6_in_9.txt` (capture **3834**)

**Problem:** no standalone stream line `6:` — chunk `6` JSON lives **inside chunk `9`'s payload**.

**Second problem:** the full description is **not** only in chunk `6` JSON. Chunk `9` also has a **`T…,plain-text`** prefix. The JSON tree references that text as **`$9`**. Rendering `textProps` alone skips `$9` → you lose the intro paragraph.

Fixtures load from repo root automatically.

In [ ]:
import json
from pathlib import Path

from adapters.linkedin import rsc
from adapters.linkedin.extract.about_the_job import extract_about_the_job

DESCRIPTION_CHUNK_ID = "6"
EMBEDDED_MARKER = f'{DESCRIPTION_CHUNK_ID}:["$"'
FLIGHT_TEXT_REF = "$9"


def find_repo_root() -> Path:
    for directory in [Path.cwd(), *Path.cwd().parents]:
        fixtures = directory / "tests/fixtures/linkedin/about_the_job"
        if fixtures.is_dir():
            return directory
    raise FileNotFoundError("Could not find tests/fixtures/linkedin/about_the_job")


EMBEDDED_FIXTURE = (
    find_repo_root() / "tests/fixtures/linkedin/about_the_job/embedded_chunk_6_in_9.txt"
)
embedded_body = EMBEDDED_FIXTURE.read_text()

## 1. Stream layout — chunk `9` holds everything

No line `6:`. Chunk `9`'s payload = **`T48c,<plain text>` + `6:[…JSON…]`**

In [ ]:
line_ids = []
for line in embedded_body.splitlines():
    line_ids.append(line.split(":", 1)[0])

print("stream line ids:", line_ids)
print("has standalone line 6:", DESCRIPTION_CHUNK_ID in line_ids)
print("get_chunk_parsed(..., '6'):", rsc.get_chunk_parsed(embedded_body, DESCRIPTION_CHUNK_ID))

chunk_9_raw = rsc.parse_stream(embedded_body)["9"]
split_at = chunk_9_raw.find(EMBEDDED_MARKER)
print(f"chunk 9 length: {len(chunk_9_raw)}, embedded JSON starts at byte {split_at}")

## 2. Two parts inside chunk `9`

| Part | Content |
|---|---|
| **T-prefix** | Intro paragraph (`We are seeking an experienced…`) |
| **Embedded `6` JSON** | Structured tree: headings, title, `$9` ref, bullet list |

In [ ]:
t_prefix_raw = chunk_9_raw[:split_at]
t_text = t_prefix_raw[t_prefix_raw.index(",") + 1 :]  # strip T48c,

print("--- T-prefix (plain text) ---")
print(t_text[:200], "...")
print()
print("--- embedded chunk 6 JSON (start) ---")
print(chunk_9_raw[split_at : split_at + 120], "...")

## 3. The JSON tree references the T-prefix as `$9`

`rsc.render_text` skips strings starting with `$` — so **`$9` is dropped** and the intro paragraph disappears.

In [ ]:
json_part = chunk_9_raw[split_at + len(DESCRIPTION_CHUNK_ID) + 1 :]
depth = 0
for index, char in enumerate(json_part):
    if char == "[":
        depth += 1
    elif char == "]":
        depth -= 1
        if depth == 0:
            chunk_6 = json.loads(json_part[: index + 1])
            break

chunk_6_json = json.dumps(chunk_6)
print("$9 appears in chunk 6 JSON:", FLIGHT_TEXT_REF in chunk_6_json)
print()

text_props_only = rsc.render_text(chunk_6[3]["textProps"]["children"])
print("render_text (no $9 resolution) — missing intro paragraph:")
print(text_props_only[:350])
print("...")
print("contains 'We are seeking':", "We are seeking" in text_props_only)

## 4. Current extractor — fails entirely

It never reaches chunk `9`; it only looks for a standalone line `6:`.

In [ ]:
try:
    extract_about_the_job(embedded_body)
except ValueError as exc:
    print(exc)

## 5. Prototype extract (notebook only)

Steps for embedded layout:

1. Find host chunk containing embedded `6:["$",…]`
2. Split T-prefix text from JSON
3. Parse embedded chunk `6`, render `textProps` **resolving `$9` → T-prefix text**

This is what `about_the_job.py` needs next.

In [ ]:
def extract_embedded_chunk_json(raw: str, chunk_id: str = DESCRIPTION_CHUNK_ID):
    marker = f"{chunk_id}:"
    start = raw.find(marker)
    if start == -1:
        return None
    json_part = raw[start + len(marker) :]
    if not json_part.startswith("["):
        return None
    depth = 0
    for index, char in enumerate(json_part):
        if char == "[":
            depth += 1
        elif char == "]":
            depth -= 1
            if depth == 0:
                return json.loads(json_part[: index + 1]), start
    return None


def t_prefix_text(host_raw: str, split_at: int) -> str:
    prefix = host_raw[:split_at]
    comma = prefix.find(",")
    if prefix.startswith("T") and comma != -1:
        return prefix[comma + 1 :]
    return prefix


def render_text_with_refs(element, refs: dict[str, str]) -> str:
    if element is None:
        return ""
    if isinstance(element, str):
        if element in refs:
            return refs[element]
        if element.startswith("$"):
            return ""
        return element
    if rsc.is_rsc_node(element):
        component_type = element[1]
        parts = []
        if component_type == "strong":
            parts.append("\n\n")
        elif component_type == "li":
            parts.append("\n\n ")
        elif component_type == "br":
            parts.append("\n")
        props = element[3]
        if isinstance(props, dict):
            for child in props.get("children", []):
                parts.append(render_text_with_refs(child, refs))
        return "".join(parts)
    if isinstance(element, list):
        return "".join(render_text_with_refs(child, refs) for child in element)
    return ""


def prototype_extract_embedded(response_body: str) -> str:
    matches = []
    for host_id, host_raw in rsc.parse_stream(response_body).items():
        parsed = extract_embedded_chunk_json(host_raw, DESCRIPTION_CHUNK_ID)
        if parsed is not None:
            matches.append((host_id, host_raw, parsed))

    if len(matches) == 0:
        raise ValueError(f"no embedded chunk {DESCRIPTION_CHUNK_ID!r}")
    if len(matches) > 1:
        raise ValueError(f"ambiguous embedded chunk {DESCRIPTION_CHUNK_ID!r}")

    host_id, host_raw, (chunk_6, split_at) = matches[0]
    intro = t_prefix_text(host_raw, split_at)
    refs = {FLIGHT_TEXT_REF: f"\n\n{intro}\n\n"}
    text_tree = chunk_6[3]["textProps"]["children"]
    print(f"host chunk {host_id}, resolved {FLIGHT_TEXT_REF} from T-prefix ({len(intro)} chars)")
    return render_text_with_refs(text_tree, refs).strip()

In [ ]:
description = prototype_extract_embedded(embedded_body)
print(f"full description: {len(description)} chars\n")
print(description[:1500])

In [ ]:
assert "Job Description" in description
assert "Practice Specialist Solution Architect" in description
assert "We are seeking an experienced Practice Specialist" in description
assert "Responsibilities" in description
assert "Lead client discussions" in description
print("checks passed — intro paragraph present between title and responsibilities")